# **Network Intrusion Detection System**
## Model Training & Evaluation

**Dataset:** NSL-KDD (Processed)  
**Problem Type:** Multi-Class Classification  
**Target Classes:** Normal, DoS, Probe, R2L, U2R

## 1. Libraries & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, 
                             confusion_matrix, 
                             f1_score, accuracy_score)
import joblib

## 2. Data Loading
> Loading preprocessed train and test datasets from EDA pipeline.

In [2]:
# Get the project's base directory
BASE_DIR = Path.cwd().parent

# Load the processed training and testing datasets
df_train = pd.read_csv(BASE_DIR / "data" / "processed" / "train_processed.csv")
df_test = pd.read_csv(BASE_DIR / "data" / "processed" / "test_processed.csv")

# Display the shape of both datasets
print(f"Train Shape: {df_train.shape}")
print(f"Test Shape:  {df_test.shape}")

Train Shape: (125973, 42)
Test Shape:  (22544, 42)


## 3. Feature & Target Split
> Separating input features (X) from target labels (y).

In [3]:
# Separate input features (X) and target variable (y) for the training dataset
X_train = df_train.drop(columns=['attack_type'])
y_train = df_train['attack_type']

# Separate input features (X) and target variable (y) for the testing dataset
X_test = df_test.drop(columns=['attack_type'])
y_test = df_test['attack_type']

# Display the shape of the training and testing feature sets
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

# Display the unique target classes in the training dataset
print(f"Classes: {y_train.unique()}")

X_train: (125973, 41)
X_test:  (22544, 41)
Classes: ['Normal' 'DoS' 'R2L' 'Probe' 'U2R']


## 4. Target Label Encoding
> XGBoost requires numeric labels. LabelEncoder converts:
> Normal→1, DoS→0, Probe→2, R2L→3, U2R→4

In [4]:
# Encode the target labels into numeric values for XGBoost
le_target = LabelEncoder()
y_train_encoded = le_target.fit_transform(y_train)
y_test_encoded = le_target.transform(y_test)

# Display the mapping between encoded values and original class labels
print("Classes Mapping:")
for i, cls in enumerate(le_target.classes_):
    print(f"  {i} → {cls}")

Classes Mapping:
  0 → DoS
  1 → Normal
  2 → Probe
  3 → R2L
  4 → U2R


## 5. Model Pipelines
> 5 algorithms defined as sklearn Pipelines.
> SVM includes StandardScaler (distance-based).
> Tree-based models do not require scaling.

| Model | Scaling | Reason |
|---|---|---|
| Decision Tree | ❌ | Tree-based |
| Random Forest | ❌ | Tree-based |
| Gradient Boosting | ❌ | Tree-based |
| XGBoost | ❌ | Tree-based |
| SVM | ✅ | Distance-based |

In [5]:
# Define machine learning pipelines for different classification models
pipelines = {
    "Decision Tree": Pipeline([
        ('model', DecisionTreeClassifier(random_state=42))
    ]),
    
    "Random Forest": Pipeline([
        ('model', RandomForestClassifier(random_state=42))
    ]),
    
    "Gradient Boosting": Pipeline([
        ('model', GradientBoostingClassifier(random_state=42))
    ]),
    
    "XGBoost": Pipeline([
        ('model', XGBClassifier(random_state=42))
    ]),
    
    # Apply feature scaling before training the Support Vector Machine model
    "Support Vector Machine": Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC())
    ])
}

## 6. Baseline Comparison — Cross Validation
> StratifiedKFold (5 splits) used to maintain class distribution.
> Metric: F1 Weighted (chosen due to class imbalance).
> Accuracy alone is misleading — U2R only 0.04% of data.

In [6]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import confusion_matrix, accuracy_score

# Define Stratified K-Fold Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("=== Cross Validation + Full Evaluation ===\n")

# Dictionaries to store evaluation results and trained models
cv_results = {}
trained_pipelines = {}

# Train and evaluate each machine learning pipeline
for name, pipeline in pipelines.items():
    
    # Perform cross-validation using the weighted F1-score
    f1_scores = cross_val_score(
        pipeline,
        X_train,
        y_train_encoded,
        cv=cv,
        scoring="f1_weighted"
    )
    
    # Train the model on the full training dataset
    pipeline.fit(X_train, y_train_encoded)
    trained_pipelines[name] = pipeline

    # Generate predictions on the test dataset
    y_pred = pipeline.predict(X_test)

    # Calculate evaluation metrics
    acc = accuracy_score(y_test_encoded, y_pred)
    f1 = f1_scores.mean()

    # Store the average cross-validation F1-score
    cv_results[name] = f1

    # Display the evaluation results
    print(f"{'='*45}")
    print(f"Model          : {name}")
    print(f"CV F1 Score    : {f1:.4f}")
    print(f"Accuracy       : {acc:.4f}")

=== Cross Validation + Full Evaluation ===

Model          : Decision Tree
CV F1 Score    : 0.9975
Accuracy       : 0.7615
Model          : Random Forest
CV F1 Score    : 0.9988
Accuracy       : 0.7500
Model          : Gradient Boosting
CV F1 Score    : 0.9972
Accuracy       : 0.7611
Model          : XGBoost
CV F1 Score    : 0.9990
Accuracy       : 0.7574
Model          : Support Vector Machine
CV F1 Score    : 0.9921
Accuracy       : 0.7531


### Baseline Results

| Model | CV F1 Score | Accuracy |
|---|---|---|
| Decision Tree | 0.9975 | 0.7615 |
| Random Forest | 0.9988 | 0.7500 |
| Gradient Boosting | 0.9972 | 0.7611 |
| **XGBoost** | **0.9990** | 0.7574 |
| SVM | 0.9921 | 0.7531 |

**Best Model: XGBoost — Highest CV F1 = 0.9990**

## 7. Best Model — Initial Evaluation
> XGBoost selected as best model based on CV F1 Score.
> Initial evaluation WITHOUT class weight adjustment.

In [7]:
# Select the best-performing trained model
best_model = trained_pipelines['XGBoost']

# Generate predictions using the best model
y_pred_best = best_model.predict(X_test)

# Display the classification report
print(classification_report(
    y_test_encoded,
    y_pred_best,
    target_names=le_target.classes_
))

# Display the confusion matrix
print(confusion_matrix(
    y_test_encoded,
    y_pred_best,
))

              precision    recall  f1-score   support

         DoS       0.96      0.80      0.87      7458
      Normal       0.66      0.97      0.79      9711
       Probe       0.81      0.68      0.74      2421
         R2L       0.83      0.01      0.02      2889
         U2R       0.71      0.15      0.25        65

    accuracy                           0.76     22544
   macro avg       0.80      0.52      0.53     22544
weighted avg       0.80      0.76      0.71     22544

[[5962 1335  161    0    0]
 [  82 9423  201    3    2]
 [ 161  609 1651    0    0]
 [   0 2844   14   29    2]
 [   0   52    0    3   10]]


### Initial Evaluation Observation
- Normal: Recall 0.97 ✅ — model identifies safe traffic well
- DoS: Precision 0.96 ✅ — strong attack detection
- **R2L: Recall 0.01 ❌** — critical failure
- **U2R: Recall 0.15 ❌** — critical failure

**Root Cause:** Class imbalance
- Train R2L = 995 vs Test R2L = 2,889 (3x mismatch!)
- Train U2R = 52 — too few examples

**Solution:** Apply class weights to penalize minority class errors more.

## 8. Class Weight — Handling Imbalance
> compute_class_weight('balanced') assigns higher weight
> to minority classes (R2L, U2R).
> This is NOT hyperparameter tuning —
> it tells the model to penalize minority class errors more.

| Class | Weight |
|---|---|
| DoS | 0.5486 |
| Normal | 0.3741 |
| Probe | 2.1615 |
| R2L | 25.3212 |
| U2R | 484.5115 |

In [8]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Calculate class weights to handle class imbalance
classes = np.unique(y_train_encoded)
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train_encoded
)

# Create a dictionary that maps each class to its corresponding weight
class_weight_dict = dict(zip(classes, weights))

# Display the calculated class weights
print("Class Weights:")
for i, cls in enumerate(le_target.classes_):
    print(f"  {cls}: {weights[i]:.4f}")

Class Weights:
  DoS: 0.5486
  Normal: 0.3741
  Probe: 2.1615
  R2L: 25.3212
  U2R: 484.5115


## 9. Final Model — XGBoost with Class Weight
> Retrained XGBoost using sample_weight parameter.
> model__sample_weight syntax used inside Pipeline.

In [9]:
# Calculate sample weights for each training instance
sample_weights = np.array([
    class_weight_dict[y] for y in y_train_encoded
])

# Create the pipeline using the XGBoost classifier
best_pipeline = Pipeline([
    ('model', XGBClassifier(random_state=42))
])

# Train the model using the calculated sample weights
best_pipeline.fit(
    X_train,
    y_train_encoded,
    model__sample_weight=sample_weights
)

# Generate predictions on the test dataset
y_pred_weighted = best_pipeline.predict(X_test)

# Display the classification report
print(classification_report(
    y_test_encoded,
    y_pred_weighted,
    target_names=le_target.classes_
))

# Display the confusion matrix
print(confusion_matrix(
    y_test_encoded,
    y_pred_weighted,
))

              precision    recall  f1-score   support

         DoS       0.96      0.77      0.85      7458
      Normal       0.67      0.97      0.79      9711
       Probe       0.87      0.75      0.81      2421
         R2L       0.99      0.12      0.21      2889
         U2R       0.62      0.20      0.30        65

    accuracy                           0.77     22544
   macro avg       0.82      0.56      0.59     22544
weighted avg       0.83      0.77      0.74     22544

[[5748 1665   45    0    0]
 [  80 9415  211    2    3]
 [ 164  433 1824    0    0]
 [   0 2514   23  347    5]
 [   0   49    1    2   13]]


### Final Evaluation — Before vs After

| Class | F1 Before | F1 After | Change |
|---|---|---|---|
| DoS | 0.87 | 0.85 | ↓ -0.02 |
| Normal | 0.79 | 0.79 | = |
| Probe | 0.74 | 0.81 | ↑ +0.07 |
| R2L | 0.02 | 0.21 | ↑ **10x** |
| U2R | 0.25 | 0.30 | ↑ +0.05 |
| **Weighted F1** | **0.71** | **0.74** | ↑ **+0.03** |

**Trade-off accepted:** DoS slightly reduced but
minority classes significantly improved.
In security systems, detecting R2L and U2R is critical.

## 10. Save Best Model
> Saving trained pipeline and label encoder using joblib.
> Both files required for FastAPI deployment.

In [10]:
import joblib

# Best model save
joblib.dump(best_pipeline, 
            BASE_DIR / "model" / "best_model.pkl")

# Label encoder save
joblib.dump(le_target, 
            BASE_DIR / "model" / "label_encoder.pkl")

print("✅ Model saved!")
print("✅ Label Encoder saved!")

✅ Model saved!
✅ Label Encoder saved!


## Model Training Complete ✅

| Item | Detail |
|---|---|
| Best Model | XGBoost |
| CV F1 Score | 0.9990 |
| Final Weighted F1 | 0.74 |
| Class Imbalance Fix | compute_class_weight balanced |
| Saved Files | best_model.pkl, label_encoder.pkl |

**Next Step:** FastAPI Deployment → `api/main.py`